# Multi-Agent · Day 36 — **Task Delegation & Shared Memory**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2.5 hrs

The supervisor (Day 33) passed everything through one hop's `State`. That breaks down when agents need to **share results across time** (a later agent reads what an earlier one wrote), **hand each other sub-tasks** (A asks B a question mid-work), and run **concurrently** without corrupting shared state. The production answer is an external store (Redis / DynamoDB) as shared memory plus a task queue as the delegation channel.

We model both with an in-memory store that has the *same API shape* as Redis, so the code you write here is a one-line swap away from `redis-py` (which you build for real in today's Python notebook).

Today:
1. A **shared memory store** — namespaced, TTL'd, the Redis API in miniature.
2. A **task queue** — orchestrator pushes, workers pull, concurrently.
3. **Agent-to-agent delegation** — agent A asks agent B a sub-question.
4. A **4-agent credit-memo workflow** end to end over the shared store.

## 1. Shared memory store (Redis-shaped)

Agents must not pass giant blobs through graph state — they write to a store keyed by `session_id` and read back by key. This mock mirrors `hset/hget/expire` so swapping in `redis-py` later changes nothing but the constructor.

In [1]:
import time, json
from typing import Any

class SharedMemory:
    """In-memory stand-in for Redis hashes + TTL. Same method names as redis-py."""
    def __init__(self):
        self._data: dict[str, dict[str, str]] = {}
        self._expiry: dict[str, float] = {}

    def _live(self, session: str) -> bool:
        exp = self._expiry.get(session)
        if exp and time.time() > exp:
            self._data.pop(session, None); self._expiry.pop(session, None)
            return False
        return True

    def hset(self, session: str, key: str, value: Any) -> None:
        self._data.setdefault(session, {})[key] = json.dumps(value)

    def hget(self, session: str, key: str) -> Any:
        if not self._live(session): return None
        raw = self._data.get(session, {}).get(key)
        return json.loads(raw) if raw is not None else None

    def hgetall(self, session: str) -> dict:
        if not self._live(session): return {}
        return {k: json.loads(v) for k, v in self._data.get(session, {}).items()}

    def expire(self, session: str, ttl_seconds: int) -> None:
        self._expiry[session] = time.time() + ttl_seconds

mem = SharedMemory()
mem.hset("sess-77", "customer", {"id": "C-4417", "name": "ACME Ltd"})
mem.expire("sess-77", 24 * 3600)                       # sessions auto-expire in 24h
print("read back:", mem.hget("sess-77", "customer"))

read back: {'id': 'C-4417', 'name': 'ACME Ltd'}


☝ Two design rules baked in. **Namespace by session** (`hset(session, key, ...)`) so concurrent customers never see each other's data — the same isolation you'd get from a Redis key prefix. **TTL everything** (`expire`) so agent working-memory is self-cleaning; a stuck workflow leaves no permanent litter. Because the method names match `redis-py`, `mem = redis.Redis()` is the only change to go from notebook to a real cluster.

## 2. Task queue — delegation channel

Delegation = the orchestrator putting sub-tasks somewhere workers pull from. You built `asyncio.Queue` on Day 33; here's the same producer/consumer with results landing back in shared memory so *any* agent can read them.

In [2]:
import asyncio, nest_asyncio
nest_asyncio.apply()          # lets asyncio.run() work inside a Jupyter kernel loop
from dataclasses import dataclass

@dataclass
class SubTask:
    session: str
    kind: str          # which specialist should handle it
    payload: dict

async def worker(name: str, q: asyncio.Queue, mem: SharedMemory,
                 handlers: dict) -> None:
    while True:
        task: SubTask = await q.get()
        try:
            result = handlers[task.kind](task.payload)
            mem.hset(task.session, f"result:{task.kind}", result)   # share the result
            print(f"  [{name}] did {task.kind} -> {result}")
        finally:
            q.task_done()                              # release q.join(), even on failure

async def run_pipeline(session: str, tasks: list[SubTask], handlers: dict):
    q: asyncio.Queue = asyncio.Queue()
    workers = [asyncio.create_task(worker(f"w{i}", q, mem, handlers)) for i in range(3)]
    for t in tasks: q.put_nowait(t)
    await q.join()                                     # block until every task_done()
    for w in workers: w.cancel()

HANDLERS = {
    "enrich":  lambda p: {"turnover": 4_200_000, "sector": "manufacturing"},
    "score":   lambda p: {"pd": 0.018, "grade": "BBB"},
    "limit":   lambda p: {"suggested_limit": 750_000},
}
tasks = [SubTask("sess-77", k, {"customer": "C-4417"}) for k in HANDLERS]
asyncio.run(run_pipeline("sess-77", tasks, HANDLERS))
print("store now:", json.dumps(mem.hgetall("sess-77"), indent=2))

  [w0] did enrich -> {'turnover': 4200000, 'sector': 'manufacturing'}
  [w0] did score -> {'pd': 0.018, 'grade': 'BBB'}
  [w0] did limit -> {'suggested_limit': 750000}
store now: {
  "customer": {
    "id": "C-4417",
    "name": "ACME Ltd"
  },
  "result:enrich": {
    "turnover": 4200000,
    "sector": "manufacturing"
  },
  "result:score": {
    "pd": 0.018,
    "grade": "BBB"
  },
  "result:limit": {
    "suggested_limit": 750000
  }
}


☝ The queue gives **fan-out for free** — three workers drain a shared queue, whoever's idle grabs the next sub-task — and results don't flow back through a return value, they land in shared memory keyed by kind. That's the decoupling that matters: the producer doesn't wait on any specific worker, and a *downstream* agent reads `result:score` whenever it needs it, regardless of who computed it or when.

## 3. Agent-to-agent delegation — A asks B a sub-question

Real collaboration isn't just fan-out; sometimes mid-task, agent A needs an answer only agent B can give. Model B as a callable the orchestrator hands to A — A delegates, blocks on the sub-answer, continues.

In [3]:
def compliance_agent(question: dict) -> dict:
    """B: answers a narrow sanctions/PEP question. Owns that knowledge alone."""
    flagged = {"Vulpes SA"}
    name = question["counterparty"]
    return {"counterparty": name, "sanctioned": name in flagged}

def analysis_agent(session: str, mem: SharedMemory, ask_compliance) -> dict:
    """A: mid-analysis, must check the counterparty before it can finish."""
    score = mem.hget(session, "result:score")
    counterparty = "Vulpes SA"
    verdict = ask_compliance({"counterparty": counterparty})     # <-- delegation
    decision = "REJECT" if verdict["sanctioned"] else "PROCEED"
    out = {"grade": score["grade"], "counterparty": counterparty,
           "compliance": verdict["sanctioned"], "decision": decision}
    mem.hset(session, "analysis", out)
    return out

print(analysis_agent("sess-77", mem, compliance_agent))

{'grade': 'BBB', 'counterparty': 'Vulpes SA', 'compliance': True, 'decision': 'REJECT'}


☝ The delegation is a **capability boundary**, not just a function call: `analysis_agent` can't and shouldn't know the sanctions list — it asks the agent that owns it. That keeps sensitive data (the flagged-parties set) confined to one agent, which is both a security property (least privilege) and a maintenance one (the list changes in one place). Passing `ask_compliance` in rather than importing it is dependency injection — swap in an A2A client (Day 35) and A is now delegating to a *remote* agent with zero change to its logic.

## 4. End-to-end: 4-agent credit-memo workflow

Ingestion → (parallel enrich/score/limit workers) → Analysis (delegates to Compliance) → Report. Everything coordinates through the one shared store; the report agent reads the whole session at the end.

In [4]:
def report_agent(session: str, mem: SharedMemory) -> str:
    s = mem.hgetall(session)
    a = s["analysis"]
    return (f"CREDIT MEMO — {s['customer']['name']} ({s['customer']['id']})\n"
            f"  Sector : {s['result:enrich']['sector']}, turnover GBP {s['result:enrich']['turnover']:,}\n"
            f"  Rating : {a['grade']}  |  PD {s['result:score']['pd']:.1%}\n"
            f"  Limit  : GBP {s['result:limit']['suggested_limit']:,}\n"
            f"  Compliance: {'FLAGGED' if a['compliance'] else 'clear'}\n"
            f"  DECISION: {a['decision']}")

# (ingestion already wrote 'customer'; workers wrote result:*; analysis wrote 'analysis')
print(report_agent("sess-77", mem))
print("\naudit keys in session:", list(mem.hgetall("sess-77").keys()))

CREDIT MEMO — ACME Ltd (C-4417)
  Sector : manufacturing, turnover GBP 4,200,000
  Rating : BBB  |  PD 1.8%
  Limit  : GBP 750,000
  Compliance: FLAGGED
  DECISION: REJECT

audit keys in session: ['customer', 'result:enrich', 'result:score', 'result:limit', 'analysis']


☝ Notice what the report agent needed: **nothing but the session id**. Four agents, some concurrent, one delegating to a fifth — and the final memo is assembled by reading one namespaced store. That's the architecture: agents are stateless workers; the *store* is the memory, the *queue* is the delegation channel, and the session id is the thread that ties a run together and makes it auditable end to end. In production, `SharedMemory` becomes Redis (24h working memory) or DynamoDB (durable), and the `expire` calls stop the store growing without bound.

## 5. Recap

| Need | Mechanism | Cell |
|---|---|---|
| Share results across agents/time | namespaced store (Redis-shaped `hset/hget`) | 1 |
| Auto-clean working memory | `expire` / TTL per session | 1 |
| Fan-out delegation | task queue, N concurrent workers | 2 |
| A asks B a sub-question | inject B as a callable; capability boundary | 3 |
| Keep sensitive data confined | B owns the list; A only asks | 3 |
| Auditable end-to-end run | one session id ties every write together | 4 |

**The seam that scales:** stateless agents + external shared memory + a queue. Tomorrow (Day 37): what happens when a worker *fails* mid-pipeline — retries with backoff, circuit breakers, and graceful partial failure.